<a href="https://colab.research.google.com/github/Digital-AI-Finance/Cryptocurrency/blob/main/02_cryptography_security/notebooks/NB01_Cryptography_Code_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lesson 2: Cryptography and Security

This notebook covers cryptographic concepts essential to cryptocurrencies:
- SHA-256 hashing and the avalanche effect
- Digital signatures using ECDSA (Elliptic Curve Digital Signature Algorithm)
- Public/private key pairs
- Wallet address generation

These are the security primitives that protect cryptocurrency transactions!

## Learning Objectives
By the end of this notebook, you will:
- Compute SHA-256 hashes and observe the avalanche effect
- Generate ECDSA key pairs using the secp256k1 curve
- Create and verify digital signatures
- Derive cryptocurrency wallet addresses from public keys
- Understand the security primitives protecting blockchain transactions

## Setup

Install the `ecdsa` library for elliptic curve cryptography.

In [ ]:
!pip install -q ecdsa

In [ ]:
import hashlib
from ecdsa import SigningKey, VerifyingKey, SECP256k1, BadSignatureError
from typing import Tuple

print("All imports successful!")

## Part 1: SHA-256 Hashing

Hashing is a one-way function that converts any input into a fixed-size output (256 bits for SHA-256).

Key properties:
- **Deterministic**: Same input always produces same output
- **Fast to compute**: Quick to calculate the hash
- **Avalanche effect**: Tiny input change = completely different hash
- **One-way**: Impossible to reverse (hash → original data)

In [ ]:
def sha256_hash(data: str) -> str:
    """Compute SHA-256 hash of the input string."""
    return hashlib.sha256(data.encode()).hexdigest()

# Example
message = "Hello, Blockchain!"
hash_value = sha256_hash(message)

print(f"Message: {message}")
print(f"SHA-256 Hash: {hash_value}")
print(f"Hash length: {len(hash_value)} characters (64 hex = 256 bits)")

## Part 2: Avalanche Effect Demonstration

Even a tiny change (one character) produces a completely different hash.

In [ ]:
def demonstrate_avalanche():
    """Show how small changes produce completely different hashes."""
    text1 = "Hello, Blockchain!"
    text2 = "Hello, Blockchain."  # Only difference: ! vs .

    hash1 = sha256_hash(text1)
    hash2 = sha256_hash(text2)

    print("=" * 70)
    print("AVALANCHE EFFECT DEMONSTRATION")
    print("=" * 70)
    print(f"\nText 1: '{text1}'")
    print(f"Hash 1: {hash1}")
    print(f"\nText 2: '{text2}'")
    print(f"Hash 2: {hash2}")

    # Count differing bits
    diff_bits = bin(int(hash1, 16) ^ int(hash2, 16)).count('1')
    print(f"\nDiffering bits: {diff_bits} out of 256 ({diff_bits/256*100:.1f}%)")
    print("\nNotice: ~50% of bits changed from a single character difference!")
    print("=" * 70)

demonstrate_avalanche()

## Part 3: Determinism Demonstration

The same input always produces the same hash.

In [ ]:
def demonstrate_determinism():
    """Show that the same input always produces the same hash."""
    text = "Cryptocurrency rocks!"

    print("\n" + "=" * 70)
    print("DETERMINISM DEMONSTRATION")
    print("=" * 70)
    print(f"\nInput: '{text}'")

    for i in range(3):
        print(f"Hash attempt {i+1}: {sha256_hash(text)}")

    print("\nNote: All hashes are identical (deterministic)")
    print("=" * 70)

demonstrate_determinism()

### TODO Exercise 1

Experiment with hashing:
1. Hash your name
2. Hash your name with a single character changed
3. Compare the two hashes - how different are they?

In [ ]:
# TODO: Hash your own data
# my_text = "Your text here"
# print(sha256_hash(my_text))

## Part 4: Digital Signatures with ECDSA

Digital signatures prove that a message was signed by the owner of a private key.

Process:
1. **Generate key pair**: Private key (secret) + Public key (shareable)
2. **Sign**: Use private key to sign a message
3. **Verify**: Anyone can verify the signature using the public key

In [ ]:
def create_keypair():
    """Generate a new ECDSA key pair."""
    private_key = SigningKey.generate(curve=SECP256k1)
    public_key = private_key.get_verifying_key()
    return private_key, public_key

def sign_message(private_key: SigningKey, message: str) -> bytes:
    """Sign a message using the private key."""
    message_hash = hashlib.sha256(message.encode()).digest()
    signature = private_key.sign_digest(message_hash, 
                                       sigencode=lambda r, s, order: r.to_bytes(32, 'big') + s.to_bytes(32, 'big'))
    return signature

def verify_signature(public_key: VerifyingKey, message: str, signature: bytes) -> bool:
    """Verify a signature using the public key."""
    try:
        message_hash = hashlib.sha256(message.encode()).digest()
        public_key.verify_digest(signature, message_hash, 
                                sigdecode=lambda sig, order: (int.from_bytes(sig[:32], 'big'), 
                                                             int.from_bytes(sig[32:], 'big')))
        return True
    except BadSignatureError:
        return False

print("Digital signature functions defined!")

## Part 5: Digital Signature Demonstration

Let's see digital signatures in action!

In [ ]:
print("=" * 70)
print("DIGITAL SIGNATURE DEMONSTRATION")
print("=" * 70)

# Create key pair
private_key, public_key = create_keypair()
print("\n1. Generated key pair (SECP256k1 curve)")
print(f"   Private key: {private_key.to_string().hex()[:32]}...")
print(f"   Public key:  {public_key.to_string().hex()[:32]}...")

# Sign a message
message = "Transfer 10 BTC to Alice"
signature = sign_message(private_key, message)
print(f"\n2. Signed message: '{message}'")
print(f"   Signature: {signature.hex()[:32]}...")

# Verify valid signature
is_valid = verify_signature(public_key, message, signature)
print(f"\n3. Verification with correct public key: {is_valid}")

# Tamper with message
tampered_message = "Transfer 100 BTC to Alice"
is_valid_tampered = verify_signature(public_key, tampered_message, signature)
print(f"\n4. Verification with tampered message: {is_valid_tampered}")
print("   (Signature doesn't match the tampered message!)")

# Verify with wrong key
_, wrong_public_key = create_keypair()
is_valid_wrong_key = verify_signature(wrong_public_key, message, signature)
print(f"\n5. Verification with wrong public key: {is_valid_wrong_key}")
print("   (Signature doesn't match a different public key!)")

print("\n" + "=" * 70)

### TODO Exercise 2

Practice with digital signatures:
1. Create your own key pair
2. Sign a message of your choice
3. Verify it works
4. Try tampering with the message and see verification fail

In [ ]:
# TODO: Create and test your own signatures
# my_private, my_public = create_keypair()
# my_message = "Your message"
# my_sig = sign_message(my_private, my_message)
# print(verify_signature(my_public, my_message, my_sig))

## Part 6: Wallet Address Generation

Cryptocurrency addresses are derived from public keys.

Process:
1. Generate private key (random 256-bit number)
2. Derive public key (elliptic curve point)
3. Hash public key to create address

In [ ]:
def generate_keypair() -> Tuple[str, str]:
    """Generate a new ECDSA key pair."""
    private_key = SigningKey.generate(curve=SECP256k1)
    public_key = private_key.get_verifying_key()

    private_key_hex = private_key.to_string().hex()
    public_key_hex = public_key.to_string().hex()

    return private_key_hex, public_key_hex

def public_key_to_ethereum_address(public_key_hex: str) -> str:
    """Derive Ethereum-style address from public key.
    
    Process:
    1. Hash public key with SHA-256 (simplified - real Ethereum uses Keccak-256)
    2. Take last 20 bytes
    3. Add '0x' prefix
    """
    public_key_bytes = bytes.fromhex(public_key_hex)
    hash_digest = hashlib.sha256(public_key_bytes).digest()

    # Take last 20 bytes
    address_bytes = hash_digest[-20:]
    address = '0x' + address_bytes.hex()

    return address

print("Wallet functions defined!")

## Part 7: Wallet Demonstration

Let's create a complete wallet with keys and address!

In [ ]:
print("=" * 70)
print("CRYPTOCURRENCY WALLET DEMONSTRATION")
print("=" * 70)

# Generate key pair
private_key, public_key = generate_keypair()

print("\n1. GENERATED KEY PAIR")
print("-" * 70)
print(f"Private Key (hex):")
print(f"  {private_key}")
print(f"\nPublic Key (hex, uncompressed):")
print(f"  {public_key}")

# Derive Ethereum-style address
address = public_key_to_ethereum_address(public_key)

print("\n2. DERIVED ETHEREUM-STYLE ADDRESS")
print("-" * 70)
print(f"Address: {address}")

print("\n3. KEY MANAGEMENT WARNINGS")
print("-" * 70)
print("WARNING: NEVER share your private key!")
print("WARNING: Loss of private key = loss of funds (irreversible)")
print("INFO: Public key and address can be safely shared")

print("\n" + "=" * 70)

### TODO Exercise 3

Create multiple wallets and explore:
1. Generate 3 different wallets
2. Compare their addresses - are they unique?
3. Notice how addresses are much shorter than public keys

In [ ]:
# TODO: Create multiple wallets
# for i in range(3):
#     priv, pub = generate_keypair()
#     addr = public_key_to_ethereum_address(pub)
#     print(f"Wallet {i+1}: {addr}")

## Key Takeaways

1. **SHA-256 hashing** is deterministic and has strong avalanche effect
2. **Digital signatures** prove ownership without revealing private keys
3. **ECDSA** is the algorithm used by Bitcoin and Ethereum for signatures
4. **Wallet addresses** are derived from public keys via hashing
5. **Private keys must be kept secret** - they control access to funds

This cryptography is what makes cryptocurrency secure and trustless!